In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Multi-Document Policy RAG — Retrieval-Augmented Generation With Citations

## 1. Project Overview

**Task:** Build a RAG system that ingests multiple policy documents, retrieves the most relevant chunks for any user question, generates a grounded answer **with inline citations**, shows the source chunks, and reports answer confidence.

**Approach:** Chunk each document with metadata → embed with a local sentence-transformer → store in ChromaDB → retrieve with metadata filters → generate grounded answers with ChatOllama.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store with metadata filtering
- `LangChain` — text splitting, retrieval orchestration

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Multi-document ingestion** — load and chunk several documents, preserving source metadata |
| 2 | **Metadata-aware retrieval** — filter search by document name, section, or category |
| 3 | **Citation generation** — produce answers with [Source: doc, §section] references |
| 4 | **Source chunk display** — show the exact chunks that informed each answer |
| 5 | **Answer confidence estimation** — LLM self-assessment of how well sources support the answer |
| 6 | **ChromaDB metadata filtering** — `where` and `where_document` filters |
| 7 | **Evaluate RAG quality** — faithfulness, relevance, and citation accuracy |

## 3. Problem Statement

Organizations maintain dozens of policy documents — HR policies, IT security, travel, expense, compliance, data privacy. Employees ask questions that span multiple documents:

- *"Can I expense a client dinner abroad?"* → Expense Policy + Travel Policy
- *"What happens if I lose my company laptop?"* → IT Security + Asset Management
- *"How many sick days do I get and do I need a doctor's note?"* → Leave Policy + HR Handbook

A naive single-document search misses cross-document answers. A RAG system with metadata filtering retrieves from the right documents and cites its sources.

## 4. Why This Project Matters

- **Real enterprise need** — policy Q&A chatbots are one of the top RAG use cases
- **Multi-document retrieval** is harder than single-document — requires metadata and ranking
- **Citations** build trust — users need to know *where* an answer came from
- **Confidence notes** tell users when the system is uncertain and they should verify manually

## 5. Pipeline Architecture

```
Policy Documents (HR, IT, Travel, Expense, Leave)
         │
    ┌────┴─────────────────────────────────────┐
    │  Ingestion Pipeline                       │
    │  1. Load each document                    │
    │  2. Split into chunks (500 chars, 100 ov) │
    │  3. Tag with metadata:                    │
    │     • doc_name, section, category          │
    │  4. Embed (all-MiniLM-L6-v2)              │
    │  5. Store in ChromaDB                     │
    └──────────────────────────────────────────┘
         │
    User Question
         │
         ▼
    ┌──────────────────────────────┐
    │  Retrieval                    │
    │  • Optional metadata filter   │
    │  • Similarity search (top-k)  │
    │  • Return chunks + metadata   │
    └──────────────────────────────┘
         │
         ▼
    ┌──────────────────────────────┐
    │  Generation                   │
    │  • Grounded answer            │
    │  • Inline [Source] citations   │
    │  • Confidence assessment       │
    └──────────────────────────────┘
```

### Why Multi-Document Retrieval Is Different

| Aspect | Single-Document RAG | Multi-Document RAG |
|--------|--------------------|--------------------|  
| **Source tracking** | Not needed | Essential — must cite which doc |
| **Metadata** | Optional | Required — doc name, section, category |
| **Filtering** | N/A | Filter by doc category before similarity search |
| **Conflicting info** | Impossible | Common — policies may contradict each other |
| **Chunk diversity** | Chunks are similar | Need chunks from different docs for cross-doc questions |

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 5              # Chunks to retrieve per query
TEMP_ANSWER = 0.1      # Low temperature for grounded answers
TEMP_JUDGE = 0.0       # Deterministic for evaluation

print("Configuration")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Chunk: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval: top-{TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Policy Documents & Ingestion

## 10. Sample Policy Documents

We define 5 realistic company policy documents with sections. In production these would be loaded from PDFs/Word docs.

In [ ]:
POLICY_DOCS = {
    "HR Handbook": {
        "category": "hr",
        "sections": {
            "Employment Types": (
                "The company recognizes three employment types: full-time (40 hours/week), "
                "part-time (20-30 hours/week), and contract (project-based). Full-time employees "
                "are eligible for all benefits including health insurance, retirement contributions, "
                "and stock options after a 90-day probation period. Part-time employees receive "
                "pro-rated benefits. Contract workers do not receive company benefits but may "
                "negotiate a higher hourly rate to compensate."
            ),
            "Performance Reviews": (
                "Performance reviews are conducted bi-annually in June and December. Each review "
                "includes a self-assessment, manager assessment, and peer feedback from at least "
                "two colleagues. Ratings use a 5-point scale: Exceeds Expectations (5), Meets "
                "Expectations (3-4), Needs Improvement (2), and Unsatisfactory (1). Employees "
                "rated Unsatisfactory for two consecutive cycles may be placed on a Performance "
                "Improvement Plan (PIP) lasting 60 days. Employees rated Exceeds Expectations "
                "are eligible for a bonus of 10-20% of base salary."
            ),
            "Code of Conduct": (
                "All employees must maintain professional conduct in the workplace. Harassment, "
                "discrimination, and bullying are strictly prohibited. Violations should be "
                "reported to HR or through the anonymous ethics hotline at ext. 4400. The company "
                "maintains a zero-tolerance policy for retaliation against reporters. First-time "
                "minor violations result in a written warning; repeated or serious violations "
                "may lead to immediate termination."
            ),
        },
    },
    "Leave Policy": {
        "category": "hr",
        "sections": {
            "Annual Leave": (
                "Full-time employees accrue 20 days of paid annual leave per year, prorated for "
                "the first year of employment. Unused leave may carry over up to a maximum of 5 "
                "days into the next calendar year; any excess is forfeited on January 1. Leave "
                "requests must be submitted at least 5 business days in advance via the HR portal. "
                "Managers may deny leave during critical business periods (quarter-end, product "
                "launches) with written justification. Part-time employees receive prorated leave "
                "based on their contracted hours."
            ),
            "Sick Leave": (
                "Employees are entitled to 10 days of paid sick leave per year. Sick leave does "
                "not carry over and resets on January 1. For absences of 3 or more consecutive "
                "days, a medical certificate is required upon return. Employees must notify their "
                "manager by 9:00 AM on the day of absence via email or phone. Unused sick leave "
                "is not paid out upon termination. The company reserves the right to request a "
                "fitness-for-duty evaluation for extended or frequent sick leave usage."
            ),
            "Parental Leave": (
                "Primary caregivers are entitled to 16 weeks of paid parental leave. Secondary "
                "caregivers receive 4 weeks of paid leave. Parental leave may be taken within 12 "
                "months of the birth or adoption date and may be split into a maximum of 2 blocks. "
                "Employees must provide at least 30 days notice before the expected start date. "
                "During parental leave, all benefits continue and the employee's position (or an "
                "equivalent role) is guaranteed upon return."
            ),
        },
    },
    "Expense Policy": {
        "category": "finance",
        "sections": {
            "General Rules": (
                "All business expenses must be submitted within 30 days of incurrence with "
                "itemized receipts. Expenses over $100 require manager pre-approval. The company "
                "reimburses reasonable and necessary business expenses only. Personal expenses "
                "mixed with business expenses must be clearly separated. Falsification of expense "
                "reports is considered fraud and grounds for immediate termination."
            ),
            "Meals and Entertainment": (
                "Business meals with clients are reimbursable up to $75 per person for lunch and "
                "$150 per person for dinner. Alcohol is reimbursable up to $30 per person when "
                "accompanying a meal with clients. Internal team meals require VP-level approval "
                "and are limited to $40 per person. Tips should not exceed 20% of the pre-tax "
                "bill. All meal receipts must include the names of attendees and business purpose."
            ),
            "Travel Expenses": (
                "Flights must be booked at least 14 days in advance through the company travel "
                "portal. Economy class is standard for flights under 6 hours; business class may "
                "be approved for flights over 6 hours with director-level approval. Hotel rates "
                "must not exceed the per-diem rate for the destination city (available on the "
                "finance intranet). Rental cars require manager approval and should be mid-size "
                "or smaller. Ride-sharing (Uber/Lyft) is preferred over taxis and rental cars for "
                "local transport. International travel requires VP approval at least 21 days in "
                "advance."
            ),
        },
    },
    "IT Security Policy": {
        "category": "it",
        "sections": {
            "Password Requirements": (
                "All passwords must be at least 14 characters long and include uppercase, "
                "lowercase, numbers, and special characters. Passwords must be changed every 90 "
                "days and cannot repeat any of the last 12 passwords. Multi-factor authentication "
                "(MFA) is mandatory for all systems. Sharing passwords is strictly prohibited "
                "and constitutes a security violation. Password managers are recommended; the "
                "company provides a licensed copy of 1Password for all employees."
            ),
            "Data Classification": (
                "Company data is classified into four tiers: Public (freely shareable), Internal "
                "(employees only), Confidential (need-to-know basis), and Restricted (board and "
                "executive only). All documents must be clearly labeled with their classification "
                "level. Confidential and Restricted data must be encrypted at rest and in transit. "
                "Sharing Confidential data externally requires written CISO approval. Any "
                "unauthorized disclosure of Restricted data is a terminable offense and may "
                "result in legal action."
            ),
            "Device Management": (
                "All company devices must be enrolled in the MDM (Mobile Device Management) "
                "system within 24 hours of issuance. Devices must have full-disk encryption "
                "enabled, automatic screen lock after 5 minutes, and the latest OS security "
                "patches installed within 48 hours of release. Lost or stolen devices must be "
                "reported to IT Security within 1 hour at security@company.com or ext. 5500. "
                "IT Security will remotely wipe the device within 2 hours of the report. "
                "Employees are financially responsible for negligent loss of company devices, "
                "up to a maximum of $500."
            ),
        },
    },
    "Remote Work Policy": {
        "category": "hr",
        "sections": {
            "Eligibility": (
                "All full-time employees who have completed the 90-day probation period are "
                "eligible for remote work. Remote work arrangements must be approved by the "
                "employee's direct manager and HR. Certain roles (facilities, lab technicians, "
                "on-site support) are exempt from remote work eligibility. Employees may work "
                "remotely up to 3 days per week unless their team has a different arrangement "
                "approved by the department head."
            ),
            "Equipment and Connectivity": (
                "The company provides a one-time $500 stipend for home office setup (desk, chair, "
                "monitor). Employees must maintain a reliable internet connection of at least "
                "25 Mbps download / 5 Mbps upload. The company reimburses up to $50/month for "
                "internet costs with receipt submission. All remote work must be performed on "
                "company-issued devices enrolled in the MDM system. Personal devices may not be "
                "used to access Confidential or Restricted data."
            ),
            "Work Hours and Availability": (
                "Remote employees must be available during core hours of 10:00 AM to 3:00 PM in "
                "their local timezone. Outside core hours, employees may flex their schedule as "
                "long as they complete their contracted 40 hours per week. All remote employees "
                "must respond to messages within 30 minutes during core hours. Cameras should be "
                "on during team meetings and client calls. Working from a location outside the "
                "employee's registered home address for more than 5 consecutive days requires "
                "manager and HR approval due to tax and compliance implications."
            ),
        },
    },
}

print(f"Loaded {len(POLICY_DOCS)} policy documents")
for doc_name, doc in POLICY_DOCS.items():
    sections = list(doc["sections"].keys())
    total_chars = sum(len(v) for v in doc["sections"].values())
    print(f"  [{doc['category'].upper():>7}] {doc_name}: {len(sections)} sections, {total_chars:,} chars")

## 11. Chunk Documents With Metadata

Each chunk carries metadata: `doc_name`, `section`, `category`. This metadata enables filtered retrieval later.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", ", ", " "],
)

all_chunks = []     # list of {"text": ..., "metadata": {...}}

for doc_name, doc in POLICY_DOCS.items():
    category = doc["category"]
    for section_name, section_text in doc["sections"].items():
        # Prepend section header for context
        full_text = f"{doc_name} — {section_name}\n\n{section_text}"
        chunks = splitter.split_text(full_text)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "metadata": {
                    "doc_name": doc_name,
                    "section": section_name,
                    "category": category,
                    "chunk_index": i,
                },
            })

print(f"Total chunks: {len(all_chunks)}")
print(f"\nChunk breakdown by document:")
from collections import Counter
doc_counts = Counter(c["metadata"]["doc_name"] for c in all_chunks)
for doc_name, count in doc_counts.items():
    print(f"  {doc_name}: {count} chunks")

# Show a sample chunk
sample = all_chunks[5]
print(f"\nSample chunk:")
print(f"  Doc: {sample['metadata']['doc_name']}")
print(f"  Section: {sample['metadata']['section']}")
print(f"  Category: {sample['metadata']['category']}")
print(f"  Text: {sample['text'][:150]}...")

## 12. Build ChromaDB Vector Store

In [ ]:
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")

# Initialize ChromaDB
chroma_client = chromadb.Client()  # In-memory for this notebook

# Delete existing collection if it exists (for re-runs)
try:
    chroma_client.delete_collection("policy_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="policy_docs",
    metadata={"hnsw:space": "cosine"},
)

# Embed and add all chunks
texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

# Embed in batches
batch_size = 32
all_embeds = []
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    batch_embeds = embeddings.embed_documents(batch)
    all_embeds.extend(batch_embeds)
    print(f"  Embedded batch {i // batch_size + 1}/{(len(texts) - 1) // batch_size + 1}")

collection.add(
    documents=texts,
    embeddings=all_embeds,
    metadatas=metadatas,
    ids=ids,
)

print(f"\n✅ ChromaDB collection created: {collection.count()} chunks indexed")

---

# Part B — Retrieval With Metadata Filtering

## 13. Basic Retrieval (No Filter)

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             where: Optional[dict] = None) -> list[dict]:
    """
    Retrieve the most relevant chunks for a query.
    Optionally filter by metadata (where clause).
    Returns list of {text, metadata, distance}.
    """
    query_embed = embeddings.embed_query(query)

    kwargs = {
        "query_embeddings": [query_embed],
        "n_results": top_k,
    }
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def display_chunks(chunks: list[dict]):
    """Pretty-print retrieved chunks."""
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c["distance"]  # cosine similarity
        print(f"\n  [{i}] (sim={sim:.3f}) {m['doc_name']} → {m['section']}")
        print(f"      {c['text'][:120]}...")


# Test basic retrieval
query = "How many sick days do I get?"
chunks = retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(chunks)} chunks:")
display_chunks(chunks)

## 14. Metadata Filtering

### How ChromaDB `where` Filters Work

ChromaDB supports filtering on metadata fields *before* similarity search:

```python
# Filter by exact match
where={"category": "hr"}

# Filter with operators
where={"doc_name": {"$eq": "Leave Policy"}}

# Combine filters (AND)
where={"$and": [
    {"category": "hr"},
    {"doc_name": {"$ne": "HR Handbook"}}
]}

# OR filter
where={"$or": [
    {"category": "hr"},
    {"category": "finance"}
]}
```

Filtering **narrows the search space** before computing similarity. This means:
- Faster search on large collections
- More relevant results when the user's intent is clear
- Avoids irrelevant cross-domain matches

In [ ]:
# Filter: only HR policies
query = "What are the rules about working from home?"
print("=" * 60)
print(f"Query: {query}")

print(f"\n--- No filter (all docs) ---")
chunks_all = retrieve(query)
display_chunks(chunks_all)

print(f"\n--- Filter: category = 'hr' ---")
chunks_hr = retrieve(query, where={"category": "hr"})
display_chunks(chunks_hr)

print(f"\n--- Filter: doc_name = 'Remote Work Policy' ---")
chunks_remote = retrieve(query, where={"doc_name": "Remote Work Policy"})
display_chunks(chunks_remote)

## 15. Cross-Document Retrieval

Some questions span multiple documents. The retriever should return chunks from different sources.

In [ ]:
cross_doc_queries = [
    "Can I expense a client dinner while traveling internationally?",
    "What equipment does the company provide and what are the security requirements?",
    "What happens if I lose my company laptop while working remotely?",
]

print("CROSS-DOCUMENT RETRIEVAL")
print("=" * 60)

for q in cross_doc_queries:
    chunks = retrieve(q)
    docs_hit = set(c["metadata"]["doc_name"] for c in chunks)
    print(f"\n  Q: {q}")
    print(f"  Docs retrieved: {', '.join(docs_hit)}")
    for c in chunks[:3]:
        m = c["metadata"]
        sim = 1 - c["distance"]
        print(f"    [{sim:.3f}] {m['doc_name']} → {m['section']}")

---

# Part C — Answer Generation With Citations

## 16. Grounded Answer Generator

In [ ]:
RAG_SYSTEM = """You are a company policy assistant. Answer employee questions using
ONLY the provided source documents. Follow these rules strictly:

1. Base your answer ONLY on the provided sources. Do not use outside knowledge.
2. Include inline citations in the format [Source: DocName, §Section].
3. If the sources don't contain enough information, say so explicitly.
4. If sources from different documents provide complementary information,
   synthesize them and cite each source.
5. Be specific — quote numbers, limits, and deadlines from the sources."""

RAG_PROMPT = """Answer this employee question using the source documents below.

QUESTION: {question}

SOURCE DOCUMENTS:
{sources}

Return ONLY JSON:
{{
  "answer": "Your grounded answer with [Source: DocName, §Section] citations",
  "confidence": "high|medium|low",
  "confidence_reason": "Why you rated confidence this way",
  "sources_used": ["DocName — Section", ...],
  "sources_not_helpful": ["DocName — Section", ...]
}}"""


def format_sources(chunks: list[dict]) -> str:
    """Format retrieved chunks as numbered sources for the prompt."""
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        parts.append(
            f"[Source {i}: {m['doc_name']}, §{m['section']}]\n{c['text']}\n"
        )
    return "\n".join(parts)


def ask_policy(question: str, where: Optional[dict] = None) -> dict:
    """
    Full RAG pipeline: retrieve → format → generate → parse.
    Returns {answer, confidence, sources, chunks}.
    """
    # Retrieve
    chunks = retrieve(question, where=where)
    sources_text = format_sources(chunks)

    # Generate
    resp = ask(
        RAG_PROMPT.format(question=question, sources=sources_text),
        system=RAG_SYSTEM,
        temperature=TEMP_ANSWER,
    )
    result = parse_json(resp)

    if result:
        result["chunks"] = chunks
    else:
        result = {"answer": resp, "confidence": "unknown", "chunks": chunks}

    return result


print("ask_policy() pipeline defined")
print("Usage: result = ask_policy('How many vacation days do I get?')")

## 17. Single-Document Question

In [ ]:
q1 = "How many sick days do I get, and do I need a doctor's note?"

result_1 = ask_policy(q1)

print("SINGLE-DOCUMENT QUESTION")
print("=" * 60)
print(f"Q: {q1}")
print(f"\n{'─'*60}")
print(f"Confidence: {result_1.get('confidence', '?').upper()}")
print(f"Reason: {result_1.get('confidence_reason', '')}")
print(f"\nAnswer:")
wrap_print(result_1.get("answer", ""))

print(f"\n{'─'*60}")
print("Sources used:")
for s in result_1.get("sources_used", []):
    print(f"  ✅ {s}")
for s in result_1.get("sources_not_helpful", []):
    print(f"  ⬜ {s} (not helpful)")

## 18. Cross-Document Question

In [ ]:
q2 = "Can I expense a client dinner while traveling internationally? What are the limits?"

result_2 = ask_policy(q2)

print("CROSS-DOCUMENT QUESTION")
print("=" * 60)
print(f"Q: {q2}")
print(f"\n{'─'*60}")
print(f"Confidence: {result_2.get('confidence', '?').upper()}")
print(f"Reason: {result_2.get('confidence_reason', '')}")
print(f"\nAnswer:")
wrap_print(result_2.get("answer", ""))

print(f"\n{'─'*60}")
print("Sources used:")
for s in result_2.get("sources_used", []):
    print(f"  ✅ {s}")

## 19. Source Chunk Display

Show the exact chunks that informed the answer so users can verify.

In [ ]:
def display_answer_with_sources(question: str, result: dict):
    """Display a full answer report with source chunks."""
    print(f"\n{'═'*70}")
    print(f"  Q: {question}")
    print(f"{'═'*70}")

    # Confidence badge
    conf = result.get("confidence", "unknown")
    badge = {"high": "🟢 HIGH", "medium": "🟡 MEDIUM", "low": "🔴 LOW"}.get(conf, "⚪ ?")
    print(f"\n  Confidence: {badge}")
    print(f"  {result.get('confidence_reason', '')}")

    # Answer
    print(f"\n  {'─'*66}")
    print(f"  ANSWER:")
    print(f"  {'─'*66}")
    for line in result.get("answer", "").split("\n"):
        print(f"  {textwrap.fill(line, width=66)}")

    # Source chunks
    chunks = result.get("chunks", [])
    print(f"\n  {'─'*66}")
    print(f"  SOURCE CHUNKS ({len(chunks)} retrieved):")
    print(f"  {'─'*66}")
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c.get("distance", 1)
        print(f"\n  [{i}] {m['doc_name']} → §{m['section']}  (similarity: {sim:.3f})")
        wrapped = textwrap.fill(c["text"], width=64)
        for wl in wrapped.split("\n"):
            print(f"      {wl}")


# Display the cross-document answer with full sources
display_answer_with_sources(q2, result_2)

## 20. Filtered Question (Scoped to One Policy)

In [ ]:
q3 = "What are the password requirements?"

result_3 = ask_policy(q3, where={"category": "it"})

print("FILTERED QUESTION — IT Policies Only")
display_answer_with_sources(q3, result_3)

## 21. Out-of-Scope Question (Confidence Test)

Ask something the policies don't cover. A good system should report LOW confidence.

In [ ]:
q4 = "What is the company's policy on cryptocurrency investments for employees?"

result_4 = ask_policy(q4)

print("OUT-OF-SCOPE QUESTION")
print("=" * 60)
print(f"Q: {q4}")
print(f"\nConfidence: {result_4.get('confidence', '?').upper()}")
print(f"Reason: {result_4.get('confidence_reason', '')}")
print(f"\nAnswer:")
wrap_print(result_4.get("answer", ""))

conf = result_4.get("confidence", "")
print(f"\n{'─'*60}")
if conf == "low":
    print("✅ Correctly identified low confidence for out-of-scope question")
else:
    print(f"⚠ Expected 'low' confidence, got '{conf}' — model may be over-confident")

---

# Part D — Evaluation & Quality

## 22. RAG Quality Evaluation

In [ ]:
EVAL_SYSTEM = """You evaluate RAG (Retrieval-Augmented Generation) answer quality.
Be critical and precise. Score each dimension 1-5."""

EVAL_PROMPT = """Evaluate this RAG answer.

QUESTION: {question}
ANSWER: {answer}
SOURCE CHUNKS USED:
{sources}

Score 1-5 on each dimension:
- FAITHFULNESS: Does the answer ONLY contain information from the sources? (5=fully grounded)
- RELEVANCE: Does the answer address the question directly? (5=perfectly relevant)
- COMPLETENESS: Does the answer cover all relevant info from the sources? (5=nothing missed)
- CITATION_ACCURACY: Are citations correctly matched to the right source? (5=all correct)

Return ONLY JSON:
{{"faithfulness": N, "relevance": N, "completeness": N, "citation_accuracy": N,
  "feedback": "one sentence of feedback"}}"""

test_questions = [
    "How many sick days do I get, and do I need a doctor's note?",
    "Can I expense a client dinner while traveling internationally?",
    "What equipment does the company provide for remote work and what are the security rules?",
    "How does the performance review process work?",
]

print("RAG QUALITY EVALUATION")
print("=" * 60)

eval_results = []
for q in test_questions:
    result = ask_policy(q)
    sources_text = "\n".join(
        f"- [{c['metadata']['doc_name']}, §{c['metadata']['section']}] {c['text'][:100]}..."
        for c in result.get("chunks", [])[:3]
    )

    eval_resp = ask(
        EVAL_PROMPT.format(
            question=q,
            answer=result.get("answer", "")[:500],
            sources=sources_text,
        ),
        system=EVAL_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(eval_resp)

    if scores:
        avg = sum(scores.get(k, 0) for k in
                  ["faithfulness", "relevance", "completeness", "citation_accuracy"]) / 4
        eval_results.append({"question": q, "scores": scores, "avg": avg})
        print(f"\n  Q: {q[:60]}...")
        print(f"    Faith: {scores.get('faithfulness',0)}/5  "
              f"Rel: {scores.get('relevance',0)}/5  "
              f"Comp: {scores.get('completeness',0)}/5  "
              f"Cite: {scores.get('citation_accuracy',0)}/5  "
              f"→ Avg: {avg:.1f}/5")
        print(f"    {scores.get('feedback', '')}")

if eval_results:
    overall = sum(e["avg"] for e in eval_results) / len(eval_results)
    print(f"\n{'─'*60}")
    print(f"Overall average: {overall:.1f}/5")

## 23. Retrieval Relevance Test

Verify that the top retrieved chunks actually match the expected policy documents.

In [ ]:
retrieval_tests = [
    {
        "query": "What is the parental leave policy?",
        "expected_docs": ["Leave Policy"],
        "expected_section": "Parental Leave",
    },
    {
        "query": "What are the rules for booking flights?",
        "expected_docs": ["Expense Policy"],
        "expected_section": "Travel Expenses",
    },
    {
        "query": "What happens if I report harassment?",
        "expected_docs": ["HR Handbook"],
        "expected_section": "Code of Conduct",
    },
    {
        "query": "What are the data classification levels?",
        "expected_docs": ["IT Security Policy"],
        "expected_section": "Data Classification",
    },
    {
        "query": "Do I need to be available at specific times when working from home?",
        "expected_docs": ["Remote Work Policy"],
        "expected_section": "Work Hours and Availability",
    },
]

print("RETRIEVAL RELEVANCE TEST")
print("=" * 60)

correct = 0
for test in retrieval_tests:
    chunks = retrieve(test["query"], top_k=3)
    top_doc = chunks[0]["metadata"]["doc_name"]
    top_section = chunks[0]["metadata"]["section"]

    doc_match = top_doc in test["expected_docs"]
    sec_match = top_section == test["expected_section"]

    status = "✅" if (doc_match and sec_match) else ("🟡" if doc_match else "❌")
    if doc_match:
        correct += 1

    print(f"\n  {status} Q: {test['query'][:55]}")
    print(f"      Expected: {test['expected_docs'][0]} → {test['expected_section']}")
    print(f"      Got:      {top_doc} → {top_section}")

print(f"\n{'─'*60}")
print(f"Retrieval accuracy (doc match): {correct}/{len(retrieval_tests)} "
      f"({correct/len(retrieval_tests)*100:.0f}%)")

## 24. Multi-Document Retrieval Explained

### What Is Multi-Document Retrieval?

In single-document RAG, all chunks come from one source. The retriever just ranks by similarity. In multi-document RAG, chunks from *different* documents compete for the top-k slots:

```
Query: 'Can I expense a client dinner while traveling?'

Candidate chunks (ranked by similarity):
  1. [0.89] Expense Policy → Meals and Entertainment  ← from doc 1
  2. [0.85] Expense Policy → Travel Expenses          ← from doc 1
  3. [0.72] Expense Policy → General Rules             ← from doc 1
  4. [0.68] Leave Policy → Annual Leave                ← irrelevant
  5. [0.65] Remote Work Policy → Equipment             ← irrelevant
```

### The Problem: Diversity vs. Relevance

Sometimes you *want* chunks from multiple documents:

```
Query: 'What equipment do I get and what are the security rules?'

Ideal retrieval:
  1. Remote Work Policy → Equipment and Connectivity
  2. IT Security Policy → Device Management
  3. IT Security Policy → Password Requirements
```

### Metadata Filtering as a Solution

| Strategy | When to Use | ChromaDB Syntax |
|----------|------------|----------------|
| **No filter** | General questions spanning all docs | `retrieve(query)` |
| **Category filter** | User asks about a known domain (HR, IT, finance) | `where={"category": "hr"}` |
| **Document filter** | User refers to a specific policy | `where={"doc_name": "Leave Policy"}` |
| **Section filter** | Drilling into a specific topic | `where={"section": "Sick Leave"}` |
| **Negative filter** | Excluding irrelevant docs | `where={"doc_name": {"$ne": "..."}}` |

### Advanced: Automatic Filter Selection

In production, an **intent classifier** can detect which policy category the user is asking about and apply the appropriate filter automatically — before the similarity search runs.

## 25. Automatic Category Detection + Filtered RAG

In [ ]:
CLASSIFY_PROMPT = """Classify this employee question into one or more policy categories.

Categories: hr, finance, it
- hr: employment, leave, performance, conduct, remote work
- finance: expenses, meals, travel, reimbursement
- it: security, passwords, devices, data classification

Question: {question}

Return ONLY JSON:
{{"categories": ["cat1", "cat2"], "reason": "why these categories"}}"""


def smart_ask_policy(question: str) -> dict:
    """
    Automatically detect relevant category and apply metadata filter.
    Falls back to no filter if multiple categories are detected.
    """
    # Step 1: Classify
    cls_resp = ask(CLASSIFY_PROMPT.format(question=question), temperature=0.0)
    cls = parse_json(cls_resp)

    where_filter = None
    if cls and len(cls.get("categories", [])) == 1:
        category = cls["categories"][0]
        where_filter = {"category": category}
        print(f"  🔍 Auto-filter: category = '{category}' ({cls.get('reason', '')})")
    elif cls and len(cls.get("categories", [])) > 1:
        cats = cls["categories"]
        where_filter = {"$or": [{"category": c} for c in cats]}
        print(f"  🔍 Auto-filter: categories = {cats}")
    else:
        print(f"  🔍 No filter applied (classification unclear)")

    # Step 2: RAG
    return ask_policy(question, where=where_filter)


# Test
print("SMART RAG — Automatic Category Detection")
print("=" * 60)

smart_qs = [
    "How many days off do I get as a new employee?",
    "Can I claim an Uber ride to the airport for a business trip?",
    "What should I do if my company phone is stolen?",
]

for q in smart_qs:
    print(f"\nQ: {q}")
    result = smart_ask_policy(q)
    print(f"  Confidence: {result.get('confidence', '?').upper()}")
    answer = result.get("answer", "")
    print(f"  Answer: {textwrap.shorten(answer, width=120, placeholder='...')}")
    print(f"  Sources: {', '.join(result.get('sources_used', []))}")

## 26. Chunk Size Experiment

In [ ]:
def test_chunk_size(chunk_size: int, query: str, expected_doc: str) -> dict:
    """Re-chunk, re-embed, and test retrieval at a given chunk size."""
    splitter_test = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_size // 5,
    )

    test_chunks = []
    for doc_name, doc in POLICY_DOCS.items():
        for sec_name, sec_text in doc["sections"].items():
            full = f"{doc_name} — {sec_name}\n\n{sec_text}"
            for i, ch in enumerate(splitter_test.split_text(full)):
                test_chunks.append({
                    "text": ch,
                    "metadata": {"doc_name": doc_name, "section": sec_name,
                                 "category": doc["category"]},
                })

    # Create temp collection
    coll_name = f"test_{chunk_size}"
    try:
        chroma_client.delete_collection(coll_name)
    except Exception:
        pass
    coll = chroma_client.create_collection(coll_name, metadata={"hnsw:space": "cosine"})

    texts_t = [c["text"] for c in test_chunks]
    embeds_t = embeddings.embed_documents(texts_t)
    coll.add(
        documents=texts_t,
        embeddings=embeds_t,
        metadatas=[c["metadata"] for c in test_chunks],
        ids=[f"c{i}" for i in range(len(test_chunks))],
    )

    q_embed = embeddings.embed_query(query)
    res = coll.query(query_embeddings=[q_embed], n_results=3)
    top_doc = res["metadatas"][0][0]["doc_name"]
    top_sim = 1 - res["distances"][0][0]

    # Cleanup
    chroma_client.delete_collection(coll_name)

    return {
        "chunk_size": chunk_size,
        "num_chunks": len(test_chunks),
        "top_doc": top_doc,
        "top_sim": top_sim,
        "correct": top_doc == expected_doc,
    }


test_query = "What happens if I lose my company laptop?"
expected = "IT Security Policy"

print("CHUNK SIZE EXPERIMENT")
print("=" * 60)
print(f"Query: {test_query}")
print(f"Expected top doc: {expected}")
print()

for size in [200, 500, 800, 1200]:
    r = test_chunk_size(size, test_query, expected)
    status = "✅" if r["correct"] else "❌"
    print(f"  {status} chunk_size={r['chunk_size']:>5} → {r['num_chunks']:>3} chunks | "
          f"top: {r['top_doc']} (sim={r['top_sim']:.3f})")

## 27. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **No source metadata** | Can't cite which document an answer came from | Tag every chunk with doc_name, section, category |
| **Flat retrieval (no filters)** | HR questions retrieve IT policy chunks by coincidence | Use metadata filters to scope search by category or doc |
| **Chunks too large** | Exceeds embedding model's context; mixes unrelated topics | 300-600 chars per chunk with 20% overlap |
| **Chunks too small** | Loses context; fragments sentences | At least 200 chars; include section headers in each chunk |
| **No confidence signal** | Users trust every answer equally, even wrong ones | Ask the LLM to self-assess confidence based on source coverage |
| **No source display** | Users can't verify answers; no accountability | Show the exact retrieved chunks alongside the answer |
| **Stuffing too many chunks** | Exceeds LLM context window; dilutes relevant info | top-k=3-5 is usually optimal; more is rarely better |
| **Ignoring conflicts** | Two policies may contradict; answer picks one silently | Detect conflicting sources and surface both to the user |

## 28. Production Improvement Ideas

1. **Hybrid search** — combine keyword (BM25) and semantic (embedding) retrieval for better recall
2. **Re-ranking** — use a cross-encoder to re-rank the top-20 chunks down to top-5 before generation
3. **Document versioning** — track policy version numbers so answers cite "v2.3 (Jan 2026)"
4. **Conflict detection** — if two chunks give contradictory info, surface both and flag the conflict
5. **Conversational memory** — maintain chat history so follow-up questions resolve coreferences
6. **Admin dashboard** — show which questions couldn't be answered (low confidence) to identify policy gaps
7. **PDF ingestion pipeline** — use pypdf or unstructured.io to extract text from real policy PDFs
8. **Access control** — some policies are visible only to managers; filter retrieved chunks by user role

## 29. Exercises

### Exercise 1: Add a New Policy Document

In [ ]:
# ── Exercise 1: Add a Data Privacy Policy ─────────────
# Add this policy to POLICY_DOCS, re-chunk, and re-index.
# Then test with privacy-related questions.

# POLICY_DOCS["Data Privacy Policy"] = {
#     "category": "compliance",
#     "sections": {
#         "Data Retention": "Customer data is retained for 3 years...",
#         "GDPR Compliance": "EU user data must be stored in EU regions...",
#         "Data Subject Rights": "Users can request data deletion within 30 days...",
#     },
# }
# # Then re-run cells 11-12 to re-index.

print("Exercise 1: Add a Data Privacy Policy and test with GDPR questions.")
print("Steps: define policy → re-chunk → re-embed → test retrieval")

### Exercise 2: Detect Conflicting Policies

In [ ]:
# ── Exercise 2: Cross-reference check ─────────────────

CONFLICT_PROMPT = """Check if these source chunks contain any conflicting or
contradictory information.

Sources:
{sources}

Return ONLY JSON:
{{
  "has_conflict": true/false,
  "conflicts": [
    {{"source_a": "doc/section", "source_b": "doc/section",
      "description": "what the conflict is"}}
  ],
  "consistent_points": ["points that all sources agree on"]
}}"""

# Test: retrieve chunks about equipment/devices from multiple docs
q_conflict = "What devices and equipment policies does the company have?"
chunks_c = retrieve(q_conflict, top_k=6)
sources_c = format_sources(chunks_c)

conflict_resp = ask(
    CONFLICT_PROMPT.format(sources=sources_c),
    temperature=TEMP_JUDGE,
)
conflict = parse_json(conflict_resp)

print("CONFLICT DETECTION")
print("=" * 60)
print(f"Q: {q_conflict}")
print(f"Docs searched: {set(c['metadata']['doc_name'] for c in chunks_c)}")

if conflict:
    print(f"\nConflicts found: {conflict.get('has_conflict', False)}")
    for c_item in conflict.get("conflicts", []):
        print(f"  ⚠ {c_item.get('source_a', '')} vs {c_item.get('source_b', '')}")
        print(f"    {c_item.get('description', '')}")
    for p in conflict.get("consistent_points", []):
        print(f"  ✅ {p}")
else:
    wrap_print(conflict_resp[:300])

### Exercise 3: Batch Question Answering

In [ ]:
# ── Exercise 3: Answer a batch of common employee questions ──

faq_questions = [
    "How do I submit an expense report?",
    "Can I carry over unused vacation days?",
    "How often do performance reviews happen?",
    "What's the maximum I can spend on a client lunch?",
    "How long is parental leave for the primary caregiver?",
    "Do I need MFA for all systems?",
    "Can I use my personal laptop for work?",
    "How do I report workplace harassment?",
]

print("BATCH FAQ ANSWERS")
print("=" * 60)

for q in faq_questions:
    result = ask_policy(q)
    answer = result.get("answer", "")
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")

    print(f"\n  {badge} Q: {q}")
    print(f"     A: {textwrap.shorten(answer, width=100, placeholder='...')}")

### Mini Challenge

1. **Hybrid retrieval:** Implement a simple keyword filter (check if the query contains domain-specific keywords like "password", "leave", "expense") and use it as a pre-filter before embedding search. Compare retrieval accuracy vs. pure semantic search.

2. **Confidence calibration:** Generate answers for 10 questions where you know the ground truth. Compare the LLM's self-assessed confidence (high/medium/low) against whether it actually answered correctly. Is the model well-calibrated?

3. **Citation verification:** For each generated answer, extract the citations and programmatically verify they reference chunks that were actually retrieved. Compute citation precision (% of citations that match a real source).

4. **Policy gap detection:** Collect all low-confidence answers from a batch of 20 questions. Cluster the unanswered topics — these represent gaps in the policy documents that should be addressed.

## 30. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nDocuments indexed: {len(POLICY_DOCS)}")
print(f"  HR Handbook, Leave Policy, Expense Policy,")
print(f"  IT Security Policy, Remote Work Policy")
print(f"Total chunks: {len(all_chunks)}")

print(f"\nCapabilities demonstrated:")
print(f"  ✅ Multi-document ingestion with metadata")
print(f"  ✅ ChromaDB storage with cosine similarity")
print(f"  ✅ Metadata filtering (category, doc_name, section)")
print(f"  ✅ Cross-document retrieval")
print(f"  ✅ Grounded answers with inline [Source] citations")
print(f"  ✅ Source chunk display for verification")
print(f"  ✅ Answer confidence estimation (high/medium/low)")
print(f"  ✅ Automatic category detection + filtered RAG")
print(f"  ✅ Out-of-scope detection")
print(f"  ✅ RAG quality evaluation (faithfulness, relevance, completeness, citation)")
print(f"  ✅ Retrieval accuracy testing")
print(f"  ✅ Chunk size experiment")
print(f"  ✅ Conflict detection across documents")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Tag every chunk with metadata** (doc_name, section, category) — citations depend on it |
| 2 | **Metadata filters** narrow the search space and improve relevance for domain-specific questions |
| 3 | **Inline citations** [Source: Doc, §Section] make answers verifiable and trustworthy |
| 4 | **Always display source chunks** — users need to see what the LLM read to trust the answer |
| 5 | **Confidence estimation** (high/medium/low) signals when the system should defer to a human |
| 6 | **Cross-document retrieval** requires diverse chunk retrieval — not all top-k from one doc |
| 7 | **Auto-classification** before retrieval improves precision by scoping the search |
| 8 | **Chunk size matters** — 400-600 chars is the sweet spot for policy text |
| 9 | **Conflict detection** is critical when multiple policies may overlap or contradict |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*